# CUPED — Variance Reduction in A/B Testing

Demonstrates CUPED (Controlled-experiment Using Pre-Experiment Data) on a
synthetic e-commerce A/B test. CUPED uses pre-experiment behaviour to reduce
the variance of the treatment effect estimate — giving tighter confidence
intervals without collecting more data.

| | |
|---|---|
| **Treatment** | `treatment` — randomly assigned (0 = control, 1 = new feature) |
| **Outcome** | `post_revenue` — revenue per user during the experiment |
| **Covariate** | `pre_revenue` — revenue per user in the 30 days before the experiment |
| **True effect** | +$2.00 per user (small lift we want to detect) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

RNG = np.random.default_rng(42)

## Generate synthetic data

We simulate 10,000 users. Each user has a latent 'spending propensity' that
drives both their pre-experiment and post-experiment revenue — this is what
creates the correlation that CUPED exploits.

The true treatment effect is **+$2.00**. We want to see whether the standard
A/B test can reliably detect it, and whether CUPED helps.

In [ ]:
N           = 10_000   # number of users
TRUE_EFFECT = 2.0      # true ATT in dollars
CORRELATION = 0.7      # pre/post revenue correlation — key CUPED parameter

# Latent spending propensity shared across both periods
propensity = RNG.normal(0, 1, N)

# Pre-experiment revenue: driven by propensity + noise, log-scaled to look like revenue
pre_revenue = np.exp(3 + 0.8 * propensity + RNG.normal(0, 0.5, N))

# Random treatment assignment — 50/50 split
treatment = RNG.binomial(1, 0.5, N)

# Post-experiment revenue: correlated with pre via shared propensity, plus true treatment effect
post_revenue = (
    np.exp(3 + 0.8 * propensity + RNG.normal(0, 0.5, N))
    + TRUE_EFFECT * treatment
)

df = pd.DataFrame({
    "user_id":     np.arange(N),
    "treatment":   treatment,
    "pre_revenue":  pre_revenue,
    "post_revenue": post_revenue,
})

print(f"Users         : {N:,}")
print(f"Treatment     : {treatment.sum():,} ({treatment.mean():.1%})")
print(f"Control       : {(1-treatment).sum():,} ({(1-treatment).mean():.1%})")
print(f"\nPre/post correlation: {df['pre_revenue'].corr(df['post_revenue']):.3f}")
print(f"True treatment effect: ${TRUE_EFFECT:.2f}")

df.describe().T